#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
DATA_BUCKET = PARENT / "server/scripts/adaptive_hexsearch/out/places-processed"
L2S = [f for f in DATA_BUCKET.rglob("*.csv") if f.is_file()]

#### Merge

In [2]:
L2_DFS = []
for f in L2S:
    try: L2_DFS.append(pd.read_csv(f))
    except: L2_DFS.append(pd.DataFrame())
df_l2 = pd.concat(L2_DFS, ignore_index=True)

In [3]:
df_validity = pd.DataFrame({
    col: df_l2[col].notna().sum() / len(df_l2) for col in df_l2.columns
}, index=[0])
display(len(df_l2), df_validity)

395

,id,displayName,primaryTypeDisplayName,rating,userRatingCount,location,shortFormattedAddress,googleMapsUri,priceRange,priceLevel,websiteUri,businessStatus,types,primaryType,sourceID,seedID,Level
0,1.0,1.0,1.0,0.997468,0.997468,1.0,1.0,1.0,0.84557,0.326582,0.739241,1.0,1.0,1.0,1.0,1.0,1.0


#### Type Parsing

In [4]:
from server.scripts.clean_data.type_parsing import parse_type, check_takeaway, predict_cuisine_from_name

df_parsed = df_l2[df_l2['rating'].notna() & df_l2['userRatingCount'].notna()]
df_parsed["predictedType"] = df_parsed.apply(predict_cuisine_from_name, axis=1)
df_parsed["cuisineType"] = df_parsed.apply(parse_type, axis=1)
df_parsed["venueType"] = df_parsed.apply(check_takeaway, axis=1)

# ── Diagnostics ───────────────────────────────────────────────────────────────
dist = df_parsed["cuisineType"].value_counts()
print(f"Unique cuisineTypes : {dist.nunique()}")
unresolved = (df_parsed["cuisineType"] == "Unspecified").sum()
print(f"Still 'Unspecified'  : {unresolved} / {len(df_parsed)}  ({unresolved/len(df_parsed):.1%})")

display(df_parsed[df_parsed["cuisineType"]=="Unspecified"])


Unique cuisineTypes : 18
Still 'Unspecified'  : 0 / 394  (0.0%)


,id,displayName,primaryTypeDisplayName,rating,userRatingCount,location,shortFormattedAddress,googleMapsUri,priceRange,priceLevel,websiteUri,businessStatus,types,primaryType,sourceID,seedID,Level,predictedType,cuisineType,venueType
